# Telco Customer Churn — Preprocessing

Goal: Transform the cleaned data into a format ready for modeling.
Decisions made here are based on findings from the EDA notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

In [2]:
df = pd.read_pickle("../data/telco_customer_churn.pkl")

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 1: Drop unnecessary columns

Based on EDA findings:
- customerID: unique identifier, carries no predictive signal
- TotalCharges: 0.83 correlation with tenure, largely redundant
- gender: 26.9% churn rate, barely above overall average — weak signal
- PhoneService: 26.7% churn rate, essentially no signal

In [3]:
cols_to_drop = ['customerID', 'TotalCharges', 'gender', 'PhoneService']

df = df.drop(columns=cols_to_drop)

print(f"Columns remaining: {df.shape[1]}")
print(df.columns.tolist())

Columns remaining: 17
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn']


## Step 2: Encode binary columns

Columns with Yes/No values are mapped to 1/0 directly.
This is simpler and cleaner than one-hot encoding for binary features.

In [4]:
binary_cols = ['Partner', 'Dependents', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 
               'StreamingMovies', 'PaperlessBilling']

df[binary_cols] = df[binary_cols].apply(
    lambda col: col.map({'Yes': 1, 'No': 0, 'No internet service': 0})
)

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Binary encoding complete")
print(df[binary_cols + ['Churn']].head())

Binary encoding complete
   Partner  Dependents  OnlineSecurity  OnlineBackup  DeviceProtection  \
0        1           0               0             1                 0   
1        0           0               1             0                 1   
2        0           0               1             1                 0   
3        0           0               1             0                 1   
4        0           0               0             0                 0   

   TechSupport  StreamingTV  StreamingMovies  PaperlessBilling  Churn  
0            0            0                0                 1      0  
1            0            0                0                 0      0  
2            0            0                0                 1      1  
3            1            0                0                 0      0  
4            0            0                0                 1      1  


### Important Note: 
Some columns were not actually binary because it had 3 options: 'Yes', 'No', 'No Internet Service'. Solution: Map 'No Internet Service' to 0.

## Step 3: Encode multi-category columns

These columns have more than 2 categories so we use one-hot encoding.
drop_first=True removes one category per column to avoid the dummy 
variable trap — where one category can be perfectly predicted from 
the others, which causes problems for logistic regression.

In [5]:
multi_cols = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f"Columns after encoding: {df.shape[1]}")
print(df.columns.tolist())

Columns after encoding: 22
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Step 4: Feature engineering

Creating two new features based on EDA insights:

1. tenure_group: Churn risk is non-linear with tenure — highest in the 
   first year then drops sharply. Binning captures this lifecycle stage 
   better than a raw number.

2. num_services: Customers with more add-on services may be more 
   embedded in the product and harder to replace elsewhere — 
   potentially a retention signal.

In [6]:
df['tenure_group'] = pd.cut(df['tenure'], 
                            bins=[0, 12, 24, 48, 72],
                            labels=['0-12', '13-24', '25-48', '49-72'],
                            include_lowest=True)

df['tenure_group'] = df['tenure_group'].astype(str)
df = pd.get_dummies(df, columns=['tenure_group'], drop_first=True)

service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['num_services'] = df[service_cols].sum(axis=1)

print(f"Columns after feature engineering: {df.shape[1]}")
print(f"\nnum_services distribution:")
print(df['num_services'].value_counts().sort_index())

Columns after feature engineering: 26

num_services distribution:
num_services
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64


## Step 5: Train / test split

Splitting 80% training, 20% test.
stratify=y ensures both sets maintain the same 73.5/26.5 churn ratio —
without this, random chance could put too many churners in one set
and give us misleading results.

In [7]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:   {X_train.shape[0]} rows")
print(f"Test set:       {X_test.shape[0]} rows")
print(f"\nChurn rate in training set: {y_train.mean():.3f}")
print(f"Churn rate in test set:     {y_test.mean():.3f}")

Training set:   5634 rows
Test set:       1409 rows

Churn rate in training set: 0.265
Churn rate in test set:     0.265


## Step 6: Scale numeric features

Logistic regression is sensitive to feature scale. A feature ranging 
from 0-72 (tenure) will dominate a feature ranging from 0-6 (num_services) 
unless we standardize them.

StandardScaler transforms each numeric feature to have mean=0 and 
standard deviation=1 — putting all features on equal footing.

Important: We fit the scaler on training data only, then apply it to 
both train and test. Fitting on the full dataset would leak information 
about the test set into the model — a form of data leakage.

In [8]:
numeric_cols = ['tenure', 'MonthlyCharges', 'num_services']

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete")
print(f"\nTraining set mean after scaling:")
print(X_train[numeric_cols].mean().round(3))
print(f"\nTraining set std after scaling:")
print(X_train[numeric_cols].std().round(3))

Scaling complete

Training set mean after scaling:
tenure           -0.0
MonthlyCharges   -0.0
num_services      0.0
dtype: float64

Training set std after scaling:
tenure            1.0
MonthlyCharges    1.0
num_services      1.0
dtype: float64


## Step 7: Save processed data

Saving all four sets as pickle files for the modeling notebook.
The scaler is also saved so it can be applied consistently to 
any new data the model sees in the future.

In [9]:
X_train.to_pickle("../data/X_train.pkl")
X_test.to_pickle("../data/X_test.pkl")
y_train.to_pickle("../data/y_train.pkl")
y_test.to_pickle("../data/y_test.pkl")

with open("../data/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")
print(f"  scaler:  fitted on {len(numeric_cols)} numeric columns")

Saved:
  X_train: (5634, 25)
  X_test:  (1409, 25)
  y_train: (5634,)
  y_test:  (1409,)
  scaler:  fitted on 3 numeric columns
